In [1]:
import logging
import warnings
warnings.showwarning = lambda *args, **kwargs: None
import numpy as np
import pandas as pd
pd.set_option('display.max_colwidth', -1)
import matplotlib.pyplot as plt
from datetime import datetime
from azureml.core import Experiment, Workspace, Dataset
from azureml.train.automl import AutoMLConfig

# Setting Up The Workspace and Experiment

In [2]:
import azureml.core

experiment_name = 'AutomatedML-TimeSeriesForecasting'
ws = Workspace.from_config()
experiment = Experiment(ws, experiment_name)
output = {}
output['SDK version'] = azureml.core.VERSION
output['Subsciption ID'] = ws.subscription_id
output['Workspace'] = ws.name
output['Resource Group'] = ws.resource_group
output['Location'] = ws.location
output['Run History'] = experiment_name
output_df = pd.DataFrame(data = output, index = [''])
output_df.T

,
SDK version,1.57.0
Subsciption ID,0c19fc19-85fd-4aa4-b133-61dd20fa93df
Workspace,auotml-example-workspace
Resource Group,edwin.spartan117-rg
Location,southeastasia
Run History,AutomatedML-TimeSeriesForecasting


# Setting Up The Compute Target

In [3]:
from azureml.core.compute import ComputeTarget, AmlCompute
from azureml.core.compute_target import ComputeTargetException

cluster_name = 'AutoML-Cluster'

try:
    compute_target = ComputeTarget(workspace = ws, name = cluster_name)
    print('Using an existing compute cluster')
except ComputeTargetException:
    compute_config = AmlCompute.provisioning_configuration(vm_size = 'Standard_Ds12_v2', max_nodes = 4)
    compute_target = ComputeTarget.create(ws, cluster_name, compute_config)
    
print('Checking cluster status...')
compute_target.wait_for_completion(show_output = True, timeout_in_minutes = 20)

Using an existing compute cluster
Checking cluster status...
Succeeded
AmlCompute wait for completion finished

Minimum number of nodes requested have been provisioned


# Data Preparation

In [4]:
time_column = 'timeStamp'
target_column = 'demand'

nyc_energy = Dataset.Tabular.from_delimited_files(
    path = 'https://automlsamplenotebookdata.blob.core.windows.net/automl-sample-notebook-data/nyc_energy.csv').with_timestamp_columns(
    fine_grain_timestamp = time_column)
nyc_energy.take(10).to_pandas_dataframe().reset_index(drop = True)

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,timeStamp,demand,precip,temp
0,2012-01-01 00:00:00,4937.5,0.0,46.13
1,2012-01-01 01:00:00,4752.1,0.0,45.89
2,2012-01-01 02:00:00,4542.6,0.0,45.04
3,2012-01-01 03:00:00,4357.7,0.0,45.03
4,2012-01-01 04:00:00,4275.5,0.0,42.61
5,2012-01-01 05:00:00,4274.7,0.0,39.02
6,2012-01-01 06:00:00,4324.9,0.0,38.78
7,2012-01-01 07:00:00,4350.0,0.0,42.74
8,2012-01-01 08:00:00,4480.9,0.0,38.90
9,2012-01-01 09:00:00,4664.2,0.0,44.67


In [5]:
nyc_energy_data = nyc_energy.time_before(datetime(2017, 8, 10, 5))
nyc_energy_train = nyc_energy_data.time_before(datetime(2017, 8, 8, 5), include_boundary = True)
nyc_energy_train.to_pandas_dataframe().reset_index(drop = True).sort_values(time_column)

nyc_energy_test = nyc_energy_data.time_after(datetime(2017, 8, 8, 6))
nyc_energy_test.to_pandas_dataframe().reset_index(drop = True)

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


,timeStamp,demand,precip,temp
0,2017-08-08 06:00:00,5590.992,0.0000,66.17
1,2017-08-08 07:00:00,6147.033,0.0000,66.29
2,2017-08-08 08:00:00,6592.425,0.0000,66.72
3,2017-08-08 09:00:00,6874.533,0.0000,67.37
4,2017-08-08 10:00:00,7010.542,0.0000,68.30
5,2017-08-08 11:00:00,7078.158,0.0000,68.89
6,2017-08-08 12:00:00,7213.317,0.0000,70.60
7,2017-08-08 13:00:00,7329.750,0.0000,72.83
8,2017-08-08 14:00:00,7426.250,0.0000,73.33
9,2017-08-08 15:00:00,7505.633,0.0000,74.89


# Configuring The AutoML Task

In [6]:
automl_config = AutoMLConfig(
    task = 'forecasting', primary_metric = 'normalized_root_mean_squared_error', experiment_timeout_hours = 1, 
    training_data = nyc_energy_train, label_column_name = target_column, time_column_name = time_column, max_horizon = 24,
    compute_target = compute_target, enable_early_stopping =  True, n_cross_validations = 3, max_cores_per_iteration = -1,
    max_concurrent_iterations = 3, verbosity = logging.INFO)

experiment_run = experiment.submit(automl_config, show_output = True)

Submitting remote run.
No run_configuration provided, running on AutoML-Cluster with default configuration
Running on remote compute: AutoML-Cluster


Experiment,Id,Type,Status,Details Page,Docs Page
AutomatedML-TimeSeriesForecasting,AutoML_08297ff7-16dd-410a-bc1b-6542dce452c6,automl,NotStarted,Link to Azure Machine Learning studio,Link to Documentation



Current status: DatasetFeaturization. Beginning to featurize the CV split.
Current status: ModelSelection. Beginning model selection.

********************************************************************************************
DATA GUARDRAILS: 

TYPE:         Time Series ID detection
STATUS:       PASSED
DESCRIPTION:  The data set was analyzed, and no duplicate time index were detected.
              Learn more about time-series forecasting configurations: https://aka.ms/AutomatedMLForecastingConfiguration

********************************************************************************************

TYPE:         Short series handling
STATUS:       PASSED
DESCRIPTION:  Automated ML detected enough data points for each series in the input data to continue with training.
              Learn more about short series handling: https://aka.ms/AutomatedMLShortSeriesHandling

********************************************************************************************

TYPE:         Frequency

In [7]:
best_run, fitted_model = experiment_run.get_output()
fitted_model.steps

[('timeseriestransformer',
  TimeSeriesTransformer(country_or_region=None, drop_column_names=[], featurization_config=FeaturizationConfig(blocked_transformers=None, column_purposes=None, dataset_language=None, prediction_transform_type=None, transformer_params=None), force_time_index_features=None, freq='H', grain_column_names=['_automl_dummy_grain_col'], group=None, lookback_features_removed=False, max_horizon=24, origin_time_colname='origin', pipeline=Pipeline(memory=None, steps=[('unique_target_grain_dropper', UniqueTargetGrainDropper(cv_step_size=24, max_horizon=24, n_cross_validations=3, target_lags=[0], target_rolling_window_size=0)), ('make_numeric_na_dummies', MissingDummiesTransformer(numerical_columns=['precip', 'temp'])), ('impute_na_numeric_datetime', TimeSeriesImputer(end=None, freq='H', impute_by_horizon=False, input_column=['precip', 'temp'], limit=None, limit_direction='forward', method=OrderedDict([('ffill', [])]), option='fillna', order=None, origin=None, value={'prec

In [8]:
featurization_summary = fitted_model.named_steps['timeseriestransformer'].get_featurization_summary()
pd.DataFrame.from_records(featurization_summary)

,RawFeatureName,TypeDetected,Dropped,EngineeredFeatureCount,Transformations
0,precip,Numeric,No,2,"[MedianImputer, ImputationMarker]"
1,temp,Numeric,No,2,"[MedianImputer, ImputationMarker]"
2,_automl_target_col,Numeric,No,1,[ImputationMarker]
3,timeStamp,DateTime,No,11,[DateTimeTransformer]


# Evaluation of Time Series Forecasting Results

In [9]:
features_test = nyc_energy_test.to_pandas_dataframe().reset_index(drop = True)
demand_test = features_test.pop(target_column).values

demand_forecast, features = fitted_model.forecast(features_test)

{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe'}
{'infer_column_types': 'False', 'activity': 'to_pandas_dataframe', 'activityApp': 'TabularDataset'}


In [10]:
from common.forecasting_helper import align_outputs
from azureml.automl.runtime.shared.score.scoring import score_regression
from automl.client.core.common import constants

nyc_energy_forecast_df = align_outputs(demand_forecast, features, features_test, demand_test, target_column)

scores = score_regression(nyc_energy_forecast_df['predicted'], nyc_energy_forecast_df[target_column], 
                          list(constants.Metric.SCALAR_REGRESSION_SET), None, None, None)
scores

{'mean_absolute_error': 191.83950881140268,
 'mean_absolute_percentage_error': 2.9332863175222896,
 'normalized_root_mean_squared_error': 0.0873236350686606,
 'normalized_median_absolute_error': 0.04708408790870623,
 'root_mean_squared_log_error': 0.037601983650790656,
 'r2_score': 0.9125591097326048,
 'root_mean_squared_error': 253.62917151927348,
 'normalized_mean_absolute_error': 0.06604967070171981,
 'normalized_root_mean_squared_log_error': 0.08301223040603609,
 'explained_variance': 0.9392659602212606,
 'spearman_correlation': 0.9769865392965696,
 'median_absolute_error': 136.75447888347935}